# Colab — Train the Multiclass Transient Classifier

End-to-end workflow for running `scripts/train_multiclass_experiment.py` on
Google Colab with data hosted on Google Drive.

**Before you start:**

1. Upload your `data.zip` (~10 GB) to Google Drive at
   `My Drive/transient-astronomy/data.zip`.
2. In Colab: **Runtime → Change runtime type → GPU** (T4 is free; L4 or A100 with Pro).
3. Run the cells top to bottom.

**What the pipeline produces** (per run, under `results/experiments/<run_name>/`):

- `summary.json` — top-line test metrics + config
- `full_eval.json` — per-class precision/recall/F1 + confusion matrix
- `confusion_matrix.png` / `confusion_matrix_raw.png`
- `training_curves.png`

## 1. Sanity-check the runtime

You need a GPU. If `nvidia-smi` errors, go back to **Runtime → Change runtime type**.

In [ ]:
!nvidia-smi
import sys, torch
print('Python   :', sys.version.split()[0])
print('Torch    :', torch.__version__)
print('CUDA     :', torch.version.cuda)
print('GPU avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name :', torch.cuda.get_device_name(0))

## 2. Mount Google Drive

Colab will ask you to authorize access to your Drive. Click through the prompt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls -lh /content/drive/MyDrive/transient-astronomy/

## 3. Clone the repo

If the clone URL below is private you'll need a GitHub Personal Access Token — in that
case replace the URL with `https://<TOKEN>@github.com/lakhia13/transient-astronomy.git`.
Public repos work with the plain URL.

In [ ]:
%cd /content
!rm -rf transient-astronomy
!git clone https://github.com/lakhia13/transient-astronomy.git
%cd transient-astronomy

!git lfs install
!git lfs pull
!ls -lh checkpoints/ || true

## 4. Install dependencies

We install deps directly via `pip` (skipping `uv` + `pyproject.toml`, which insists on
Python 3.12 and Colab ships 3.11). `torch` is preinstalled, so we only add what's missing.

In [ ]:
%pip install -q \
    timm \
    albumentations \
    astropy \
    pyyaml \
    scikit-learn \
    pandas \
    matplotlib \
    seaborn \
    tqdm

## 5. Extract the dataset from Drive

The repo expects data at:

```
data/
├── labels.csv
└── processed/
    ├── split_labels.csv
    ├── stats.json
    ├── train/   (84,678 .npy files)
    ├── val/     (18,078 .npy files)
    └── test/    (18,134 .npy files)
```

If your zip extracts with an extra top-level folder (e.g. `data/data/...`) there's
a cell below that fixes it.

In [ ]:
!mkdir -p data
# Unzip takes ~5-10 min for 10 GB — be patient.
!unzip -q -o /content/drive/MyDrive/transient-astronomy/data.zip -d data/

!echo '--- top level of data/ ---'
!ls data/
!echo '--- data/processed/ ---'
!ls data/processed/ 2>/dev/null | head -20

In [ ]:
# Fix nested data/data layouts if that's what your zip produced.
import shutil
from pathlib import Path

nested = Path('data/data')
if nested.exists():
    print(f'Found nested {nested}, flattening...')
    for item in nested.iterdir():
        target = Path('data') / item.name
        if target.exists():
            print(f'  skip {item.name} (already exists at target)')
            continue
        shutil.move(str(item), str(target))
    nested.rmdir()
    print('Done.')
else:
    print('No nested data/data — layout is already correct.')

!ls data/
!ls data/processed/ 2>/dev/null | head

In [ ]:
# Verify the pipeline will find what it needs.
from pathlib import Path
import pandas as pd

labels_csv = Path('data/labels.csv')
processed_dir = Path('data/processed')

assert labels_csv.exists(), f'Missing: {labels_csv}'
assert (processed_dir / 'train').exists(), f'Missing: {processed_dir}/train'
assert (processed_dir / 'val').exists(),   f'Missing: {processed_dir}/val'
assert (processed_dir / 'test').exists(),  f'Missing: {processed_dir}/test'

df = pd.read_csv(labels_csv)
print(f'labels.csv rows : {len(df):,}')
print(f'Class counts    :\n{df["class"].value_counts()}')
for split in ('train', 'val', 'test'):
    n = len(list((processed_dir / split).glob('*.npy')))
    print(f'{split:>5} .npy files: {n:,}')

## 6. Smoke test (1 epoch)

Always run this before a full training pass. It writes to a disposable
`<run_name>_smoke` directory so a failure or Ctrl+C doesn't pollute your real results.
If this finishes cleanly in a few minutes with non-NaN metrics, the pipeline works.

In [ ]:
!python scripts/train_multiclass_experiment.py \
    --config configs/multiclass_exp_v1.yaml \
    --smoke-test

## 7. Full training run

Uses the `epochs`, `loss`, `batch_size`, etc. from
`configs/multiclass_exp_v1.yaml`. Expect **~45–90 minutes** on a T4 for 30 epochs.
The process writes a per-epoch `*_history.json` as it goes, so if Colab disconnects you
still have partial logs. The best checkpoint is saved every time validation improves.

In [ ]:
!python scripts/train_multiclass_experiment.py \
    --config configs/multiclass_exp_v1.yaml

## 8. Inspect the results

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

import yaml
cfg = yaml.safe_load(open('configs/multiclass_exp_v1.yaml'))
run_name = cfg['experiment']['name']
results_dir = Path('results/experiments') / run_name

summary = json.loads((results_dir / 'summary.json').read_text())
print(f'Run           : {summary["run_name"]}')
print(f'Best epoch    : {summary["train"]["best_epoch"]}')
print(f'Best val acc  : {summary["train"]["best_val_metric"]:.4f}')
print(f'Test accuracy : {summary["test"]["accuracy"]:.4f}')
print(f'Test macro-F1 : {summary["test"]["macro_f1"]:.4f}')
print(f'Test weighted-F1: {summary["test"]["weighted_f1"]:.4f}')
print()
print('Per-class metrics:')
for cls, m in summary['test']['per_class_metrics'].items():
    print(f'  {cls:>8} | precision={m["precision"]:.3f} | recall={m["recall"]:.3f} '
          f'| F1={m["f1"]:.3f} | n={m["support"]}')

In [ ]:
for fname in ('training_curves.png', 'confusion_matrix.png', 'confusion_matrix_raw.png'):
    p = results_dir / fname
    if p.exists():
        print(fname)
        display(Image(str(p)))
    else:
        print(f'(missing: {p})')

## 9. Save outputs back to Drive

Colab wipes `/content` on disconnect — copy everything you care about to Drive now.

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/multiclass_exp_v1.yaml'))
run_name = cfg['experiment']['name']

!mkdir -p /content/drive/MyDrive/transient-astronomy/runs
!cp -r checkpoints/experiments/{run_name} /content/drive/MyDrive/transient-astronomy/runs/
!cp -r results/experiments/{run_name}      /content/drive/MyDrive/transient-astronomy/runs/

!ls -R /content/drive/MyDrive/transient-astronomy/runs/{run_name}

## 10. (Optional) Commit the checkpoint back to GitHub via Git LFS

Easier to do this on your laptop: download
`runs/<run_name>/<run_name>_best.pt` from Drive, drop it into
`checkpoints/experiments/<run_name>/` locally, then:

```bash
brew install git-lfs     # one-time
git lfs install          # one-time
git lfs track 'checkpoints/experiments/**/*.pt'   # already covered by *.pt rule if set
git add .gitattributes checkpoints/experiments/<run_name>/
git add results/experiments/<run_name>/
git commit -m 'Add multiclass run <run_name>'
git push
```

LFS handles the large `.pt` file transparently. The JSON/PNG result files go through
normal git.